# Unlock and Test Milvus Collection

This notebook helps you:
1. Properly close existing connections
2. Query the locked collection
3. Test cross-document queries

In [ ]:
# Cell 1: Clean Up Existing Connections
import gc
from pathlib import Path

print("Cleaning up existing connections...")

# Force garbage collection to close any lingering connections
gc.collect()

print("✓ Memory cleaned")

# Check if database file exists
db_path = Path("milvus_all_docs.db")
if db_path.exists():
    print(f"✓ Database found: {db_path}")
    print(f"  Size: {db_path.stat().st_size / 1024 / 1024:.2f} MB")
else:
    print("⚠️  Database not found. Please run phase2_chunking_embedding.ipynb first.")

In [ ]:
# Cell 2: Create Fresh Connection
from pymilvus import MilvusClient

print("Creating fresh Milvus connection...\n")

# Create new client instance
milvus_client = MilvusClient(uri="./milvus_all_docs.db")
collection_name = "rag_all_docs"

# Check if collection exists
if milvus_client.has_collection(collection_name):
    print(f"✓ Collection '{collection_name}' found and accessible")
    
    # Get collection stats
    try:
        stats = milvus_client.describe_collection(collection_name)
        print(f"\nCollection Info:")
        print(f"  Name: {stats['collection_name']}")
        print(f"  Total rows: {stats['num_entities']}")
        print(f"\n✓ Collection is UNLOCKED and ready for queries!")
    except Exception as e:
        print(f"⚠️  Error accessing collection: {e}")
else:
    print(f"✗ Collection '{collection_name}' not found")
    print("Please run phase2_chunking_embedding.ipynb to create it.")

In [ ]:
# Cell 3: Load BGE Embedding Model
import torch
from sentence_transformers import SentenceTransformer

print("Loading BGE embedding model...\n")

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

embedding_model = SentenceTransformer('BAAI/bge-large-en-v1.5', device=device)
embedding_dim = 1024

print(f"✓ Model loaded")
print(f"  Dimension: {embedding_dim}")
print(f"  Device: {embedding_model.device}")

In [ ]:
# Cell 4: Test Query Function
def query_collection(question, top_k=5, filter_expr=None):
    """
    Query the Milvus collection with semantic search
    
    Args:
        question: Search query
        top_k: Number of results to return
        filter_expr: Optional filter (e.g., 'doc_type == "pdf"')
    """
    print(f"\n{'='*80}")
    print(f"QUERY: {question}")
    print(f"{'='*80}\n")
    
    # Embed query
    query_embedding = embedding_model.encode(question, normalize_embeddings=True)
    
    # Search parameters
    search_params = {
        'collection_name': collection_name,
        'data': [query_embedding.tolist()],
        'limit': top_k,
        'output_fields': ["text", "source_file", "doc_type", "page_number", 
                         "bbox_l", "bbox_t", "bbox_r", "bbox_b", "item_type"],
    }
    
    if filter_expr:
        search_params['filter'] = filter_expr
        print(f"Filter: {filter_expr}\n")
    
    # Execute search
    results = milvus_client.search(**search_params)
    
    # Display results
    print(f"Found {len(results[0])} results:\n")
    
    for i, hit in enumerate(results[0], 1):
        entity = hit['entity']
        print(f"[{i}] Score: {hit['distance']:.4f}")
        print(f"    Source: {entity['source_file']} ({entity['doc_type']})")
        print(f"    Page: {entity['page_number']}, Type: {entity['item_type']}")
        print(f"    Text: {entity['text']}...")
        print()
    
    print(f"{'='*80}\n")
    return results

print("✓ Query function defined")

In [ ]:
# Cell 5: Test Cross-Document Query (No Filter)
print("TEST 1: Query Across ALL Documents\n")
print("This should search through PDFs, DOCX, XLSX, and JPG files\n")

results = query_collection(
    question="What is the  CIN of the company?",
    top_k=10
)

In [ ]:
# Cell 6: Test Query with PDF Filter
print("TEST 2: Query Only PDF Documents\n")

results = query_collection(
    question="1.(a) Corporate Identity Number (CIN) of company",
    top_k=10,
    filter_expr='doc_type == "pdf"'
)

In [ ]:
# Cell 7: Test Query with DOCX Filter
print("TEST 3: Query Only DOCX Documents\n")

results = query_collection(
    question="audit report opinion",
    top_k=5,
    filter_expr='doc_type == "docx"'
)

In [ ]:
# Cell 8: Test Query with XLSX Filter
print("TEST 4: Query Only Excel Documents\n")

results = query_collection(
    question="balance sheet total assets",
    top_k=5,
    filter_expr='doc_type == "xlsx"'
)

In [ ]:
# Cell 9: Collection Statistics by Document Type
print("="*80)
print("COLLECTION STATISTICS")
print("="*80)

# Get all unique doc types
all_docs = milvus_client.query(
    collection_name=collection_name,
    filter="id >= 0",
    output_fields=["doc_type"],
    limit=10000
)

# Count by doc type
from collections import Counter
doc_type_counts = Counter([doc['doc_type'] for doc in all_docs])

print(f"\nTotal chunks: {len(all_docs)}")
print(f"\nChunks by document type:")
for doc_type, count in sorted(doc_type_counts.items()):
    pct = (count / len(all_docs)) * 100
    print(f"  {doc_type:10s}: {count:5d} chunks ({pct:5.1f}%)")

print(f"\n✓ Collection is accessible for cross-document queries!")
print(f"\nYou can now query across all {len(doc_type_counts)} document types simultaneously.")

In [ ]:
# Cell 10: Advanced Cross-Document Query Example
print("="*80)
print("ADVANCED: Multi-Document Query with Source Tracking")
print("="*80)

question = "authorised capital of company"
results = query_collection(question, top_k=10)

# Analyze which documents contributed to results
source_files = {}
for hit in results[0]:
    source = hit['entity']['source_file']
    if source not in source_files:
        source_files[source] = []
    source_files[source].append(hit['distance'])

print(f"\nResults came from {len(source_files)} different documents:")
for source, scores in sorted(source_files.items(), key=lambda x: max(x[1]), reverse=True):
    print(f"  {source}: {len(scores)} chunks (best score: {max(scores):.4f})")

print(f"\n✓ Successfully queried across multiple documents!")

## Summary

If you reached this point, your Milvus collection is **unlocked and fully functional**!

### What You Can Do Now:

1. **Cross-Document Queries**: Search across all PDFs, DOCX, XLSX, JPG files simultaneously
2. **Filtered Queries**: Use `filter_expr` to limit search to specific document types
3. **Metadata Access**: Get page numbers, bounding boxes, and item types for each result
4. **Source Tracking**: See which documents contributed to your query results

### Next Steps:

- Use this connection in your main application
- Integrate with Gemini for RAG-powered answers
- Build UI that highlights source locations using bbox coordinates